# Ablation Study on `ap_burst2k`

Replaces `07_ablations_and_attribution.ipynb`, which runs on `data/raw/llm_generated`. That
notebook's section 3 is the source of the claim that a gap of 0.2271 is attributable to
relational and temporal modelling -- **a claim that is refuted**, because a stricter version
of the same probe (a logistic regression on one record's embedding, no pooling, no edges, no
temporal order) also scores AUROC 1.0000 on that partition. No gap remains to attribute.

An ablation is only meaningful on a benchmark where the full model is not already saturated.
On the withdrawn benchmark every variant scored 1.0000, so nothing could be ablated. That is
why these had to be re-run.

**Each ablation changes exactly one thing** and trains through the same `Trainer` as the
detector: same loss, optimiser, batch size, augmentation, epoch budget and validation-AUROC
checkpoint selection.

| Ablation | What changes | Parameters |
|---|---|---|
| baseline | -- | 670,915 |
| seed 1 | random seed only | 670,915 |
| gated readout | attention + zero-init mean/max/count gates | 671,686 |
| mean-pool readout | `AttentionPool` -> `global_mean_pool` | 670,915 |
| no structural features | 4 per-node columns removed | fewer |
| temporal removed | `NodeMemoryModule` + `TemporalAggregator` gone | 306,243 |

## 1. Setup

In [7]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import yaml
from sklearn.metrics import roc_auc_score, average_precision_score

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
import os
os.chdir(REPO_ROOT)

from src.experiments.sequence_eval import boot_ci, paired_boot

DATA_DIR = REPO_ROOT / "data" / "raw" / "ap_burst2k"
EXP = REPO_ROOT / "experiments"
BASE = EXP / "run_burst2k_base"
CONFIG = yaml.safe_load(open(REPO_ROOT / "configs" / "default.yaml"))

# Every model is scored on the SAME 500 test sequences, in the same order. Each run's
# sequence_scores_test.npz carries its own sequence_id and y, and they are asserted equal
# before anything is paired -- comparing two models scored on different orderings produces a
# number that looks fine and means nothing.
# The reference column below is READ from experiments/reported_results.json, the frozen
# record of the figures the thesis reports, written by src/experiments/freeze_reference.py.
# It is deliberately NOT each run's live sequence_eval.json: retraining overwrites those,
# which would make the reference equal the measurement by construction and turn this into
# a check that cannot fail.
REFERENCE = json.load(open(EXP / "reported_results.json"))
REF = REFERENCE["sequence_level"]
print(f"reference: {len(REF)} results frozen at commit {REFERENCE.get('commit')} "
      f"on {REFERENCE.get('generated')}")

base = np.load(BASE / "sequence_scores_test.npz", allow_pickle=True)
Y = base["y"]
ORDER = base["sequence_id"]
DETECTOR = base["T-GNN_baseline_(window_max)"]
N_BOOT = 2000

print(f"{len(Y)} test sequences, {int(Y.sum())} poisoned")
print(f"detector (T-GNN, window max): AUROC {roc_auc_score(Y, DETECTOR):.4f}  "
      f"AP {average_precision_score(Y, DETECTOR):.4f}")


def load_scores(run_dir, key_filter=None, filename="sequence_scores_test.npz"):
    """Per-sequence scores from a run, with the ordering checked against the detector's.

    key_filter matters more than it looks. Every run directory written by sequence_eval also
    contains that run's copy of the LOGISTIC REGRESSION baselines, because sequence_eval
    computes them every time. Loading a static-GNN run without filtering therefore pulls in
    three regression rows and labels them as static-GNN results.
    """
    p = Path(run_dir) / filename
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=True)
    assert (z["sequence_id"] == ORDER).all(), f"{p} is ordered differently -- cannot pair"
    assert (z["y"] == Y).all(), f"{p} has different labels -- different split?"
    return {k: z[k] for k in z.files
            if k not in ("sequence_id", "y") and (key_filter is None or key_filter in k)}


def row(name, scores, ref_key=None, pair=True):
    """Measured, then the frozen reference, then the drift between them.

    ref_key defaults to name. Nothing is hardcoded: a row with no entry in
    reported_results.json says "no reference" rather than inventing one.
    """
    auroc, apr = roc_auc_score(Y, scores), average_precision_score(Y, scores)
    lo, hi = boot_ci(Y, scores, N_BOOT)
    out = f"{name:<38} {auroc:>7.4f} {apr:>7.4f}  [{lo:.4f}, {hi:.4f}]"
    if pair:
        d, dlo, dhi = paired_boot(Y, DETECTOR, scores, N_BOOT, metric="auroc")
        out += f"  {d:+.4f}{'*' if (dlo > 0 or dhi < 0) else ' '}"
    r = REF.get(ref_key or name)
    if r is None:
        out += "       (no reference)"
    else:
        drift = auroc - r["auroc"]
        out += (f"   {r['auroc']:.4f} {drift:+.4f}"
                + ("" if abs(drift) < 0.02 else "  <-- OUTSIDE NOISE"))
    print(out)
    return auroc, apr


HEAD = (f"{'model':<38} {'AUROC':>7} {'AP':>7}  {'95% CI':>18}  "
        f"{'dAUROC':>8}   {'ref':>7} {'drift':>7}\n" + "-" * 118)

reference: 24 results frozen at commit 86587fb on 2026-09-22
500 test sequences, 200 poisoned
detector (T-GNN, window max): AUROC 0.8122  AP 0.8185


## 2. Seed stability -- read this before any other ablation

An ablation difference is only meaningful against the run-to-run noise floor. Two independent
seeds bound it, and re-running evaluation on the *same* checkpoint in a fresh process moves
graph AUROC by about +/-0.007.

**Treat any difference below about 0.02 as noise.** Several ablations below fall inside that
band, and reporting them as effects would be wrong.

In [8]:
print(HEAD)
for label, run in (("seed 42 (baseline)", BASE),
                   ("seed 1", EXP / "run_burst2k_base_s1")):
    s = load_scores(run, key_filter="window_max")
    if not s:
        print(f"{label:<38} PENDING")
        continue
    key = list(s)[0]
    row(label, s[key], ref_key=key.replace("_", " "), pair=(run != BASE))
print("\nspread across seeds: 0.0018 AUROC. That is the noise floor -- nothing below")
print("about 0.02 should be reported as an effect.")

model                                    AUROC      AP              95% CI    dAUROC       ref   drift
----------------------------------------------------------------------------------------------------------------------
seed 42 (baseline)                      0.8122  0.8185  [0.7707, 0.8532]   0.8122 +0.0000
seed 1                                  0.8140  0.8204  [0.7733, 0.8547]  -0.0018    0.8140 +0.0000

spread across seeds: 0.0018 AUROC. That is the noise floor -- nothing below
about 0.02 should be reported as an effect.


## 3. Readout ablations

Three ways to pool node embeddings into a graph embedding.

- **attention** -- the published baseline. MIL-style attention pooling (Ilse et al., 2018).
- **gated** -- attention plus zero-initialised mean/max/count residual branches. Bit-identical
  to attention at initialisation, so it *starts* as the baseline and departs only insofar as
  training opens a gate. On `ap_burst2k` the learned gates ended at
  `[+0.0004, -0.0023, +0.0186]`: the model was free to use magnitude information and largely
  declined.
- **mean-pool** -- `AttentionPool` replaced by `global_mean_pool`, nothing else changed. This
  is the direct test of the Sections 2.6 and 3.8 claim that mean pooling averages a small
  anomalous subgraph away by construction.

**Measured: 0.6207, against attention's 0.8122.** Swapping one pooling operation costs 0.19
AUROC and 0.23 AP, and takes the model significantly *below* the 13-feature oracle
(-0.1819 AUROC, 95% CI [-0.2458, -0.1164]). The thesis argued this at length and had never
tested it on a benchmark where it could be tested -- on the withdrawn one every variant scored
1.0000. Note also that mean-pool has 662,594 parameters against attention's 670,915, so the
8,321-parameter difference cannot explain a 0.19 gap.

In [9]:
print("produced by:")
print("  gated    : python -m src.models.train --data data/raw/ap_burst2k \\")
print("               --output experiments/run_burst2k_gated --readout gated ...")
print("  mean-pool: python -m src.experiments.train_ablation_meanpool \\")
print("               --data data/raw/ap_burst2k --output experiments/run_burst2k_meanpool \\")
print("               --structural-features --skip-phase1 --epochs-supervised 12 --seed 42\n")
print(HEAD)
for label, run in (("attention (baseline)", BASE),
                   ("gated readout", EXP / "run_burst2k_gated"),
                   ("mean-pool readout", EXP / "run_burst2k_meanpool")):
    s = load_scores(run, key_filter="window_max")
    if not s:
        print(f"{label:<38} PENDING -- see the command above")
        continue
    key = list(s)[0]
    row(label, s[key], ref_key=key.replace("_", " "), pair=(run != BASE))

produced by:
  gated    : python -m src.models.train --data data/raw/ap_burst2k \
               --output experiments/run_burst2k_gated --readout gated ...
  mean-pool: python -m src.experiments.train_ablation_meanpool \
               --data data/raw/ap_burst2k --output experiments/run_burst2k_meanpool \
               --structural-features --skip-phase1 --epochs-supervised 12 --seed 42

model                                    AUROC      AP              95% CI    dAUROC       ref   drift
----------------------------------------------------------------------------------------------------------------------
attention (baseline)                    0.8122  0.8185  [0.7707, 0.8532]   0.8122 +0.0000
gated readout                           0.8016  0.8095  [0.7579, 0.8428]  +0.0104    0.8016 +0.0000
mean-pool readout                       0.6207  0.5836  [0.5708, 0.6695]  +0.1917*   0.6207 +0.0000


## 4. Input-feature ablation: structural node features

The dataset optionally emits four extra per-node columns on top of the 384-d embedding and 3
metadata columns: degree under two scalings, and the mean and max weight of a node's incident
edges. Every reported result trains with them enabled.

Removing them changes the model's input width, so the checkpoint must be evaluated with
`--no-structural-features`. Evaluating it as though it had them loads the weights into a
differently-shaped model and produces a number that means nothing -- the failure recorded in
`experiments/eval_ap_scale.INVALID_train_eval_mismatch.json`.

**Measured: 0.5078, with a 95% interval of [0.4550, 0.5623]. That is chance.** The largest
effect of any ablation here, and it needs to be reported with its explanation rather than as a
bare number, because on its own it looks alarming.

It is what the benchmark's design implies. On `ap_burst2k` the poisoned records are genuine
unmodified knowledge-base entries, so the 384-d embeddings carry no label information by
construction -- the per-record content probe scores 0.5300. The signal lives entirely in graph
structure. Strip the four columns that describe that structure and the model has nothing left.

The follow-up question is why the GAT cannot recover degree on its own, since it does see
`edge_index`. Because `GATConv` normalises its attention with a softmax, its output is a
convex combination of neighbours -- a weighted *average*, which is invariant to how many
neighbours there are. Node count and degree cannot be represented along that path and must be
supplied explicitly. This is the same limitation that motivated the gated scale readout.

**State this as a scope limitation in Section 5.7.** As evaluated on this benchmark the
detector is a structural and temporal detector; the embeddings contribute nothing, because the
benchmark was deliberately built so they could not. That is the correct design for isolating
relational modelling, and it is also a boundary on what the result claims.

In [10]:
print("produced by: python -m src.models.train --data data/raw/ap_burst2k \\")
print("               --output experiments/run_burst2k_nostruct --readout attention \\")
print("               --skip-phase1 --select-by auroc --epochs-supervised 12 --seed 42")
print("  (note: no --structural-features flag)\n")
print("evaluate with: python -m src.experiments.sequence_eval --data data/raw/ap_burst2k \\")
print("                 --run-dir experiments/run_burst2k_nostruct --no-structural-features\n")
print(HEAD)
for label, run in (("with structural features (baseline)", BASE),
                   ("without structural features", EXP / "run_burst2k_nostruct")):
    s = load_scores(run, key_filter="window_max")
    if not s:
        print(f"{label:<38} PENDING -- see the commands above")
        continue
    key = list(s)[0]
    row(label, s[key], ref_key=key.replace("_", " "), pair=(run != BASE))

produced by: python -m src.models.train --data data/raw/ap_burst2k \
               --output experiments/run_burst2k_nostruct --readout attention \
               --skip-phase1 --select-by auroc --epochs-supervised 12 --seed 42
  (note: no --structural-features flag)

evaluate with: python -m src.experiments.sequence_eval --data data/raw/ap_burst2k \
                 --run-dir experiments/run_burst2k_nostruct --no-structural-features

model                                    AUROC      AP              95% CI    dAUROC       ref   drift
----------------------------------------------------------------------------------------------------------------------
with structural features (baseline)     0.8122  0.8185  [0.7707, 0.8532]   0.8122 +0.0000
without structural features             0.5078  0.4429  [0.4550, 0.5623]  +0.3049*   0.5078 +0.0000


## 5. Component ablation: remove the temporal machinery

The largest effect in the study, and the cleanest.

`NodeMemoryModule` (the per-node GRU carrying state between snapshots) and
`TemporalAggregator` (the GRU over the snapshot sequence) are removed. The forward pass reads
`graph_sequence[-1]` and nothing else. The GAT encoder, the pooling, the head architectures,
the input width, the loss and the training recipe are all unchanged.

Run under both readouts so the loss cannot be attributed to pooling. The `window last`
variants land at 0.4813 and 0.4860 -- indistinguishable from chance, which is exactly what
the benchmark's construction predicts for a model that reads one snapshot.

In [11]:
print(HEAD)
for tag, readout in (("attn", "attention"), ("mean", "mean")):
    s = load_scores(EXP / f"run_burst2k_static_{tag}", key_filter="static_GNN")
    if not s:
        print(f"{'temporal removed (' + readout + ')':<38} PENDING")
        continue
    for agg in ("max", "mean", "last"):
        key = f"static_GNN_({readout})_(window_{agg})"
        if key in s:
            row(f"temporal removed, {readout}, {agg}", s[key],
                ref_key=key.replace("_", " "))

model                                    AUROC      AP              95% CI    dAUROC       ref   drift
----------------------------------------------------------------------------------------------------------------------
temporal removed, attention, max        0.5602  0.4684  [0.5085, 0.6106]  +0.2517*   0.5602 +0.0000
temporal removed, attention, mean       0.6178  0.5472  [0.5669, 0.6668]  +0.1941*   0.6178 +0.0000
temporal removed, attention, last       0.4813  0.3801  [0.4304, 0.5336]  +0.3307*   0.4813 +0.0000
temporal removed, mean, max             0.5424  0.4652  [0.4909, 0.5951]  +0.2700*   0.5424 +0.0000
temporal removed, mean, mean            0.5934  0.5203  [0.5413, 0.6446]  +0.2191*   0.5934 +0.0000
temporal removed, mean, last            0.4860  0.3912  [0.4355, 0.5374]  +0.3265*   0.4860 +0.0000


## 6. Ablation summary

In [12]:
runs = [("attention readout (baseline)", BASE, "window_max"),
        ("seed 1", EXP / "run_burst2k_base_s1", "window_max"),
        ("gated readout", EXP / "run_burst2k_gated", "window_max"),
        ("mean-pool readout", EXP / "run_burst2k_meanpool", "window_max"),
        ("no structural features", EXP / "run_burst2k_nostruct", "window_max"),
        ("temporal removed (attention)", EXP / "run_burst2k_static_attn", "window_max"),
        ("temporal removed (mean)", EXP / "run_burst2k_static_mean", "window_max")]

base_auroc = roc_auc_score(Y, DETECTOR)
print(f"{'ablation':<36} {'AUROC':>7} {'AP':>7} {'d vs baseline':>15}  verdict")
print("-" * 88)
pending = []
for label, run, filt in runs:
    s = load_scores(run, key_filter=filt)
    if not s:
        pending.append(label)
        print(f"{label:<36} {'PENDING':>7}")
        continue
    v = list(s.values())[0]
    auroc, apr = roc_auc_score(Y, v), average_precision_score(Y, v)
    d = auroc - base_auroc
    if run == BASE:
        verdict = "reference"
    elif abs(d) < 0.02:
        verdict = "within noise (|d| < 0.02)"
    else:
        dd, lo, hi = paired_boot(Y, DETECTOR, v, N_BOOT, metric="auroc")
        verdict = "SIGNIFICANT" if (lo > 0 or hi < 0) else "not significant"
    print(f"{label:<36} {auroc:>7.4f} {apr:>7.4f} {d:>+15.4f}  {verdict}")

if pending:
    print(f"\n{len(pending)} ablation(s) not yet measured: {', '.join(pending)}")

print("""
WHAT THIS SHOWS

Three separable effects, all significant, in order of size:

  structural node features   -0.30   removing them leaves the model AT CHANCE
  temporal components        -0.25   consistent under either readout
  attention pooling          -0.19   one operation swapped, 8,321 parameters apart

Removing the temporal components is the study's CENTRAL claim -- it is what the benchmark was
rebuilt to make measurable -- but it is not the largest effect. Say so. Claiming the central
result is also the biggest, when the table shows otherwise, is the kind of thing an examiner
checks.

The structural-features result is larger and needs its explanation attached wherever it is
quoted. On this benchmark the poisoned records are genuine unmodified knowledge-base entries,
so the embeddings carry no label information by construction and all the signal is structural.
GATConv's softmax attention produces a convex combination of neighbours -- a weighted average,
invariant to neighbour count -- so degree cannot be recovered along that path and has to be
supplied explicitly. As evaluated here the detector is a structural and temporal detector.
That is a scope limitation for Section 5.7, not a defect.

Readout choice between attention and gated, and the change of seed, both move the result by
less than the 0.02 noise floor. Do not report those as effects -- report them as evidence the
headline is stable.
""")

ablation                               AUROC      AP   d vs baseline  verdict
----------------------------------------------------------------------------------------
attention readout (baseline)          0.8122  0.8185         +0.0000  reference
seed 1                                0.8140  0.8204         +0.0018  within noise (|d| < 0.02)
gated readout                         0.8016  0.8095         -0.0106  within noise (|d| < 0.02)
mean-pool readout                     0.6207  0.5836         -0.1915  SIGNIFICANT
no structural features                0.5078  0.4429         -0.3044  SIGNIFICANT
temporal removed (attention)          0.5602  0.4684         -0.2520  SIGNIFICANT
temporal removed (mean)               0.5424  0.4652         -0.2698  SIGNIFICANT

WHAT THIS SHOWS

Three separable effects, all significant, in order of size:

  structural node features   -0.30   removing them leaves the model AT CHANCE
  temporal components        -0.25   consistent under either readout
  atten